# Reference Answer Generator
Generates reference answers for evaluation using the observation-only prompt. Runs the video through the same encoder pipeline, then calls GPT-5 with the reference prompt (no RAG context, no chat history).

In [1]:
import sys
import os
from dotenv import load_dotenv

# Add notebooks/ to path so we can import video_processing, etc.
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "notebooks"))

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

from video_processing import analyze_video
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(
    persist_directory=os.path.join(os.path.dirname(os.getcwd()), "chroma_db"),
    embedding_function=embeddings,
)

print("ChromaDB loaded:", vectorstore._collection.count(), "documents")

ChromaDB loaded: 76 documents


## Video encoder
Same as `video_encoder_node` in graph.py — extracts 15 frames, encodes as base64.

In [2]:
def video_encoder(video_path):
    """Extract 15 frames from video and encode as base64 image payloads."""
    frames = analyze_video(filepath_in=video_path, frame_count=15, max_seconds=10)
    encoded_images = [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{frame}"}}
        for frame in frames
    ]
    return encoded_images

## Reference answer generator
Uses the observation-only prompt from `reference_answer_prompt.md`. No RAG context, no chat history — just the images + prompt.

In [3]:
REFERENCE_PROMPT = open("reference_answer_prompt.md").read()

def reference_answer_generator(video_path, user_query="Analyze my form"):
    """Generate a reference answer for evaluation — observation-only, no RAG."""
    encoded_images = video_encoder(video_path)

    prompt = ChatPromptTemplate.from_messages([
        ("system", REFERENCE_PROMPT),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,
    ])

    llm = ChatOpenAI(model='gpt-5', temperature=0)
    chain = prompt | llm | StrOutputParser()

    response = chain.invoke({"input": [user_msg]})
    return response

## Test with a single video

In [11]:
VIDEO_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "raw_workout_videos")

test_video = os.path.join(VIDEO_DIR, "bench_press", "bench_20.MP4")
result = reference_answer_generator(test_video, user_query="How's my form?")
print(result)

Frames processed: 15, (10s cap)
Saved to processed-images/bench_20
Video name: bench_20
Hands set very close together on the bar.  
Thumbs are not wrapped around the bar.  
Wrists are extended backward; bar sits in the fingers.  
Forearms are not vertical at the bottom; wrists are not stacked over the elbows.  
Elbows are flared wide from the torso at the bottom.  
Bar path descends high and touches near the upper chest/neck.  
Bar finishes over the face/neck instead of over mid-chest.  
Heels are raised; feet are not flat and not firmly planted.  
Neck is extended back with the chin lifted.
